# A01 — Final Model Comparison and Statistical Evaluation

**EEEM068 Applied Machine Learning — Final Frozen-Model Test Evaluation**

**Aim:** Evaluate the frozen, selected models on the reserved internal test split and produce
the final tables, confidence intervals, statistical comparisons, and conclusions for the
report.

**Evaluation protocol — read before running:**

- All architectures, checkpoints, temperatures, and ensemble weights are **frozen**. Nothing
  in A01 changes any of them.
- **No training or model selection occurs in A01.** Every model evaluated here was already
  selected, trained, and (for M13-D) calibrated before this notebook exists.
- The internal test split is used **only** for final comparison — never to alter, retrain, or
  re-select any model.
- No test result is used to modify preprocessing, thresholds, epochs, temperatures, or
  ensemble weights.
- **QWK is the primary metric.** Macro-F1 is the principal class-balanced secondary metric.

**On the test split's integrity**: the internal test split was reserved for final model
comparison. However, some earlier baseline experiments had already reported internal-test
metrics, so it should not be described as a completely pristine holdout. In A01, all model
configurations and ensemble parameters were frozen before the comparative evaluation.

**Models evaluated as the four final systems:**

| Model | Role | Selected experiment |
|---|---|---|
| EfficientNet-B4 | Baseline | `exp01_shared_p0_wrs_focal_seed42` |
| DeiT-III | Previous coursework model | `D07_two_phase_finetuning` |
| M12 (MaxViT-Tiny) | New LSA model, best standalone MaxViT | `M12_combined_fine_grained_guidance` |
| M13-D | Final ensemble | Calibrated class-specialist ensemble over the frozen M13 candidate set |

M07 and M11 may additionally appear in a supplementary validation-only table for context, but
the main final test comparison focuses on the four systems above.

**A01 must not**: retrain any model; select a new epoch; change M13 temperatures or ensemble
weights; tune thresholds on test data; remove difficult test samples; recreate the split; use
test results to modify preprocessing; or claim the test set was completely untouched when
earlier baseline test metrics were already viewed.

**A01 is created and committed unexecuted first.** After review, it is run once, and results
are committed separately.

## 2. Reproducibility setup

In [ ]:
import json
import random
import time
from pathlib import Path
from itertools import combinations

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.nn.functional as F

from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    average_precision_score,
    classification_report,
    cohen_kappa_score,
    confusion_matrix,
    f1_score,
    log_loss,
    precision_score,
    recall_score,
)

print("Imports OK.")

In [ ]:
SEED = 42
N_BOOTSTRAP = 2000
BOOTSTRAP_SEED = 42
CI_PERCENTILES = (2.5, 97.5)  # 95% percentile CI

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

NUM_CLASSES = 5
CLASS_NAMES = [
    "No DR",
    "Mild",
    "Moderate",
    "Severe",
    "Proliferative DR",
]

PROJECT_ROOT = Path(
    "/scratch/New AML/EEEM068-LSA-Diabetic-Retinopathy"
)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print(f"Seed: {SEED}  Bootstrap seed: {BOOTSTRAP_SEED}  N_BOOTSTRAP: {N_BOOTSTRAP}")
print(f"Device: {device}")
if device.type == "cuda":
    print(f"GPU: {torch.cuda.get_device_name(0)}")
print(f"torch version: {torch.__version__}")
try:
    import timm
    print(f"timm version: {timm.__version__}")
except ImportError:
    print("timm not imported at this point (only needed if live inference runs in Section 5).")
print(f"NUM_CLASSES: {NUM_CLASSES}")
assert NUM_CLASSES == 5
print(f"CLASS_NAMES: {CLASS_NAMES}")

In [ ]:
A01_LOG_DIR = PROJECT_ROOT / "logs" / "analysis" / "A01_final_model_comparison"
A01_FIGURE_DIR = PROJECT_ROOT / "results" / "figures" / "analysis" / "A01_final_model_comparison"
for directory in [A01_LOG_DIR, A01_FIGURE_DIR]:
    directory.mkdir(parents=True, exist_ok=True)

TEST_CSV = PROJECT_ROOT / "configs" / "splits" / "internal_test_split.csv"
TRAIN_CSV = PROJECT_ROOT / "configs" / "splits" / "train_split.csv"
VAL_CSV = PROJECT_ROOT / "configs" / "splits" / "val_split.csv"

print(f"A01 logs    -> {A01_LOG_DIR}")
print(f"A01 figures -> {A01_FIGURE_DIR}")
print(f"Test split  -> {TEST_CSV}")

## 3. Frozen model manifest

In [ ]:
FROZEN_MODELS = {
    "EfficientNet-B4": {
        "role": "Baseline",
        "architecture": "EfficientNet-B4",
        "selected_experiment": "exp01_shared_p0_wrs_focal_seed42",
        "log_dir": PROJECT_ROOT / "logs" / "efficientnet_b4" / "exp01_shared_p0_wrs_focal_seed42",
        "input_size": 224,
        "preprocessing": "P0 (shared)",
        "calibration": "none",
        "ensemble_status": "standalone",
    },
    "DeiT-III": {
        "role": "Previous coursework model",
        "architecture": "DeiT3-Base/16",
        "selected_experiment": "D07_two_phase_finetuning",
        "log_dir": PROJECT_ROOT / "logs" / "deit3_b16" / "D07_two_phase_finetuning",
        "input_size": 224,
        "preprocessing": "P0",
        "calibration": "none",
        "ensemble_status": "standalone",
    },
    "M12": {
        "role": "New LSA model (best standalone MaxViT)",
        "architecture": "MaxViT-Tiny, shared 3-view backbone",
        "selected_experiment": "M12_combined_fine_grained_guidance",
        "log_dir": PROJECT_ROOT / "logs" / "maxvit_tiny" / "M12_combined_fine_grained_guidance",
        "input_size": 224,
        "preprocessing": "P0 + local crop + pseudo-mask guided view",
        "calibration": "none",
        "ensemble_status": "standalone",
    },
    "M13-D": {
        "role": "Final ensemble",
        "architecture": "Calibrated class-specialist ensemble",
        "selected_experiment": "M13_calibrated_class_specialist_ensemble",
        "log_dir": PROJECT_ROOT / "logs" / "maxvit_tiny" / "M13_calibrated_class_specialist_ensemble",
        "input_size": "n/a (combines constituent model outputs)",
        "preprocessing": "n/a (post-hoc probability fusion)",
        "calibration": "per-model temperature scaling",
        "ensemble_status": "ensemble of frozen constituent-model outputs",
    },
    # Supplementary validation-context models (not part of the main test comparison).
    "M07": {
        "role": "Supplementary (validation-only context)",
        "architecture": "MaxViT-Tiny, single global view",
        "selected_experiment": "M07_ordinal_aware_loss",
        "log_dir": PROJECT_ROOT / "logs" / "maxvit_tiny" / "M07_ordinal_aware_loss",
        "input_size": 224, "preprocessing": "P0", "calibration": "none", "ensemble_status": "standalone",
    },
    "M11": {
        "role": "Supplementary (validation-only context)",
        "architecture": "MaxViT-Tiny, shared global-local backbone",
        "selected_experiment": "M11_global_local_crop_fusion",
        "log_dir": PROJECT_ROOT / "logs" / "maxvit_tiny" / "M11_global_local_crop_fusion",
        "input_size": 224, "preprocessing": "P0 + local crop", "calibration": "none", "ensemble_status": "standalone",
    },
}

manifest_rows = []
for model_name, info in FROZEN_MODELS.items():
    manifest_rows.append({
        "Model": model_name, "Role": info["role"], "Architecture": info["architecture"],
        "Selected experiment": info["selected_experiment"], "Checkpoint/config source": str(info["log_dir"]),
        "Input size": info["input_size"], "Preprocessing": info["preprocessing"],
        "Calibration": info["calibration"], "Ensemble status": info["ensemble_status"],
    })
final_model_manifest = pd.DataFrame(manifest_rows)
print(final_model_manifest.to_string(index=False))

final_model_manifest.to_csv(A01_LOG_DIR / "final_model_manifest.csv", index=False)
print(f"\nSaved -> {A01_LOG_DIR / 'final_model_manifest.csv'}")

## 4. Test-set integrity checks

The test split is **loaded as-is from notebook 01's output** — never recreated or reshuffled
here.

In [ ]:
test_df = pd.read_csv(TEST_CSV)
train_df = pd.read_csv(TRAIN_CSV)
val_df = pd.read_csv(VAL_CSV)

required_columns = {"image", "level", "patient_id", "eye", "filepath"}
missing_columns = required_columns - set(test_df.columns)
if missing_columns:
    raise ValueError(f"Test split is missing columns: {sorted(missing_columns)}")

missing_files = test_df.loc[~test_df["filepath"].map(lambda path: Path(path).exists())]
if len(missing_files) > 0:
    raise FileNotFoundError(f"Test split references {len(missing_files):,} missing image files.")

assert test_df["image"].is_unique, "Duplicate image IDs in the test split."
assert not test_df.duplicated().any(), "Duplicate rows in the test split."
assert test_df["level"].between(0, 4).all(), "Test split contains an out-of-range label."

train_val_images = set(train_df["image"]) | set(val_df["image"])
test_images = set(test_df["image"])
assert test_images.isdisjoint(train_val_images), "Test split overlaps with train/validation images."

train_val_patients = set(train_df["patient_id"]) | set(val_df["patient_id"])
test_patients = set(test_df["patient_id"])
patient_overlap = test_patients & train_val_patients
assert len(patient_overlap) == 0, f"Test split has patient overlap with train/validation: {list(patient_overlap)[:5]}"

print(f"Test rows                : {len(test_df):,}")
print(f"Unique test image IDs    : {test_df['image'].nunique():,}")
print(f"Unique test patients     : {test_df['patient_id'].nunique():,}")
print("No duplicate image IDs   : PASSED")
print("No duplicate rows        : PASSED")
print("Labels in [0, 4]         : PASSED")
print("No image overlap w/ train/val : PASSED")
print("No patient overlap w/ train/val: PASSED")

test_set_integrity_summary = {
    "test_row_count": len(test_df),
    "unique_image_ids": int(test_df["image"].nunique()),
    "unique_patients": int(test_df["patient_id"].nunique()),
    "no_duplicate_image_ids": True,
    "no_duplicate_rows": True,
    "labels_in_range": True,
    "no_image_overlap_with_train_val": True,
    "no_patient_overlap_with_train_val": True,
    "test_csv_path": str(TEST_CSV),
}
with open(A01_LOG_DIR / "test_set_integrity_summary.json", "w") as f:
    json.dump(test_set_integrity_summary, f, indent=2)
print(f"\nSaved -> {A01_LOG_DIR / 'test_set_integrity_summary.json'}")

# The exact same test image IDs/labels must be used for every model - this reference is
# checked against every model's aligned predictions later (Section 6).
REFERENCE_TEST_IMAGE_IDS = test_df["image"].tolist()
REFERENCE_TEST_LABELS = test_df.set_index("image")["level"].to_dict()

## 5. Obtain frozen test predictions

For each of EfficientNet-B4, DeiT-III, and M12: load the selected frozen checkpoint, run
**test-only, deterministic inference** (no augmentation, `model.eval()`), and save raw
logits/probabilities/labels/image IDs plus a predictions CSV. No training happens here, no
epoch is selected, no threshold is tuned.

`get_or_run_test_inference()` first checks for an already-saved `test_logits.npy` (in case
inference was already run once and is being reused, which is exactly the frozen,
deterministic behaviour required — re-running it would be pointless and risks introducing
nondeterminism from a different environment). If not found, it calls a **model-specific**
inference function. **All five inference functions (EfficientNet-B4, DeiT-III, M07, M11,
M12) are fully implemented below** — EfficientNet-B4 loads
`checkpoints/efficientnet_b4/exp01_shared_p0_wrs_focal_seed42/best_model.pt` with `timm`'s
native `efficientnet_b4` and P0 preprocessing; DeiT-III loads D07's frozen checkpoint with
`deit3_base_patch16_224.fb_in1k`; M07, M11, and M12 reconstruct their exact architectures
from their own saved `config.json` files. None of these reconstruct logits from saved
probabilities — every model's logits come from a genuine forward pass on the actual test
images.

In [ ]:
def get_or_run_test_inference(model_name: str, log_dir: Path, inference_fn) -> dict:
    """Returns {'logits','probs','labels','image_ids'} for the test set. Loads existing
    saved outputs if present (frozen, deterministic - no need to rerun); otherwise calls
    inference_fn() to produce them, matched against REFERENCE_TEST_IMAGE_IDS."""
    frozen_dir = log_dir / "A01_frozen_test_predictions"
    logits_path = frozen_dir / "test_logits.npy"
    labels_path = frozen_dir / "test_labels.npy"
    ids_path = frozen_dir / "test_image_ids.csv"

    if logits_path.exists() and labels_path.exists() and ids_path.exists():
        logits = np.load(logits_path)
        labels = np.load(labels_path)
        image_ids = pd.read_csv(ids_path)["image_id"].astype(str).tolist()
        print(f"{model_name}: loaded existing frozen test predictions from {frozen_dir}")
    else:
        print(f"{model_name}: no existing frozen test predictions found; running inference now...")
        result = inference_fn()
        logits, labels, image_ids = result["logits"], result["labels"], result["image_ids"]

        frozen_dir.mkdir(parents=True, exist_ok=True)
        np.save(logits_path, logits)
        np.save(labels_path, labels)
        pd.DataFrame({"image_id": image_ids}).to_csv(ids_path, index=False)
        print(f"{model_name}: saved frozen test predictions -> {frozen_dir}")

    assert len(image_ids) == len(set(image_ids)), (
        f"{model_name} contains duplicate image IDs."
    )

    assert len(image_ids) == len(REFERENCE_TEST_IMAGE_IDS)

    assert set(image_ids) == set(REFERENCE_TEST_IMAGE_IDS), (
        f"{model_name}'s test image IDs do not match the reference test split exactly."
    )

    probs = F.softmax(torch.tensor(logits, dtype=torch.float64), dim=1).numpy()
    return {"logits": logits, "probs": probs, "labels": labels, "image_ids": image_ids}

In [ ]:
def run_m12_test_inference() -> dict:
    """Genuine deterministic test inference for M12, reusing this project's own M12
    architecture (shared 3-view MaxViT backbone) and P0/local-crop/pseudo-mask pipeline.
    Loads the frozen best.pt checkpoint only - does not train, does not select an epoch."""
    import cv2
    import timm
    from PIL import Image
    from tqdm.auto import tqdm
    import torchvision.transforms.functional as TF

    m12_checkpoint_dir = PROJECT_ROOT / "checkpoints" / "maxvit_tiny" / "M12_combined_fine_grained_guidance"
    m12_config_path = FROZEN_MODELS["M12"]["log_dir"] / "config.json"
    if not m12_config_path.exists():
        raise FileNotFoundError(
            f"M12 config not found at {m12_config_path} - cannot reconstruct its exact "
            "architecture/normalisation settings for frozen test inference."
        )
    with open(m12_config_path) as f:
        m12_config = json.load(f)

    IMAGE_SIZE = m12_config["image_size"]
    VIT_MEAN = tuple(m12_config["normalisation_mean"])
    VIT_STD = tuple(m12_config["normalisation_std"])
    MODEL_NAME = m12_config["model_name"]
    guidance_cfg = m12_config["guidance_pipeline"]

    class CombinedGuidanceMaxViT(nn.Module):
        def __init__(self, model_name, num_classes, dropout):
            super().__init__()
            self.backbone = timm.create_model(model_name, pretrained=False, num_classes=0, global_pool="avg")
            feature_dim = self.backbone.num_features
            self.classifier = nn.Sequential(nn.Dropout(dropout), nn.Linear(feature_dim * 3, num_classes))

        def forward(self, global_image, local_image, guided_image):
            combined = torch.cat([global_image, local_image, guided_image], dim=0)
            features = self.backbone(combined)
            g, l, gd = features.chunk(3, dim=0)
            return self.classifier(torch.cat([g, l, gd], dim=1))

    model = CombinedGuidanceMaxViT(MODEL_NAME, NUM_CLASSES, m12_config["classifier_dropout"]).to(device)
    checkpoint = torch.load(m12_checkpoint_dir / "best.pt", map_location=device, weights_only=False)
    model.load_state_dict(checkpoint["model_state"])
    model.eval()
    print(f"M12: loaded frozen checkpoint from epoch {checkpoint['epoch']} "
          f"(val_qwk={checkpoint['val_qwk']:.4f}) - test-only inference, no training.")

    # Re-use the exact same deterministic P0/candidate-detection/local-crop/guided-view
    # pipeline as M09-M12's own notebooks (identical thresholds from guidance_cfg).
    DETECTION_SIZE = guidance_cfg["detection_size"]
    CLAHE_CLIP_LIMIT = guidance_cfg["clahe_clip_limit"]
    CLAHE_TILE_GRID_SIZE = tuple(guidance_cfg["clahe_tile_grid_size"])
    BACKGROUND_SIGMA = guidance_cfg["background_sigma"]
    DARK_RESPONSE_PERCENTILE = guidance_cfg["dark_response_percentile"]
    MIN_COMPONENT_AREA = guidance_cfg["min_component_area"]
    MAX_COMPONENT_AREA = guidance_cfg["max_component_area"]
    MAX_ASPECT_RATIO = guidance_cfg["max_aspect_ratio"]
    MAX_CANDIDATE_COMPONENTS = guidance_cfg["max_candidate_components"]
    LOCAL_CROP_SIZE = guidance_cfg["local_crop_size"]
    GUIDED_VIEW_BACKGROUND_ATTENUATION = guidance_cfg["guided_view_background_attenuation"]

    def preprocess_p0_full(image_path):
        img_bgr = cv2.imread(str(image_path))
        img = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)
        gray = cv2.cvtColor(img, cv2.COLOR_RGB2GRAY)
        _, mask = cv2.threshold(gray, 10, 255, cv2.THRESH_BINARY)
        kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (25, 25))
        mask = cv2.morphologyEx(mask, cv2.MORPH_CLOSE, kernel)
        contours, _ = cv2.findContours(mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
        if contours:
            x, y, w, h = cv2.boundingRect(max(contours, key=cv2.contourArea))
            margin = int(min(w, h) * 0.02)
            x, y = max(0, x - margin), max(0, y - margin)
            w = min(img.shape[1] - x, w + 2 * margin)
            h = min(img.shape[0] - y, h + 2 * margin)
            img = img[y:y + h, x:x + w]
        h, w = img.shape[:2]
        side = max(h, w)
        canvas = np.zeros((side, side, 3), dtype=img.dtype)
        canvas[(side - h) // 2:(side - h) // 2 + h, (side - w) // 2:(side - w) // 2 + w] = img
        return canvas

    def detect_and_crop_and_mask(image_full):
        detection_image = cv2.resize(image_full, (DETECTION_SIZE, DETECTION_SIZE), interpolation=cv2.INTER_AREA)
        green = detection_image[:, :, 1]
        clahe = cv2.createCLAHE(clipLimit=CLAHE_CLIP_LIMIT, tileGridSize=CLAHE_TILE_GRID_SIZE)
        enhanced_green = clahe.apply(green)
        background = cv2.GaussianBlur(enhanced_green, (0, 0), BACKGROUND_SIGMA)
        dark_response = cv2.subtract(background, enhanced_green)
        gray = cv2.cvtColor(detection_image, cv2.COLOR_RGB2GRAY)
        field_mask = (gray > 10).astype(np.uint8)
        dark_response = dark_response * field_mask
        field_pixels = dark_response[field_mask > 0]
        if field_pixels.size == 0 or field_pixels.max() <= 0:
            binary = np.zeros_like(field_mask, dtype=np.uint8)
        else:
            threshold_value = float(np.percentile(field_pixels, DARK_RESPONSE_PERCENTILE))
            binary = ((dark_response >= threshold_value) & (dark_response > 0) & (field_mask > 0)).astype(np.uint8) * 255
        kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (3, 3))
        binary = cv2.morphologyEx(binary, cv2.MORPH_OPEN, kernel)
        num_labels, labels_arr, stats, centroids = cv2.connectedComponentsWithStats(binary, connectivity=8)

        candidates = []
        for label_id in range(1, num_labels):
            area = stats[label_id, cv2.CC_STAT_AREA]
            w, h = stats[label_id, cv2.CC_STAT_WIDTH], stats[label_id, cv2.CC_STAT_HEIGHT]
            if area < MIN_COMPONENT_AREA or area > MAX_COMPONENT_AREA:
                continue
            if max(w, h) / max(1, min(w, h)) > MAX_ASPECT_RATIO:
                continue
            score = float(dark_response[labels_arr == label_id].mean())
            cx, cy = centroids[label_id]
            candidates.append({"label_id": label_id, "score": score, "cx": cx, "cy": cy})
        candidates.sort(key=lambda c: c["score"], reverse=True)
        accepted = candidates[:MAX_CANDIDATE_COMPONENTS]

        mask_512 = np.zeros_like(binary, dtype=np.uint8)
        for c in accepted:
            mask_512[labels_arr == c["label_id"]] = 255
        pseudo_mask = cv2.resize(mask_512.astype(np.float32) / 255.0, (IMAGE_SIZE, IMAGE_SIZE), interpolation=cv2.INTER_AREA)
        pseudo_mask = np.clip(pseudo_mask, 0, 1).astype(np.float32)

        if accepted:
            cx, cy = accepted[0]["cx"], accepted[0]["cy"]
        else:
            cx, cy = DETECTION_SIZE / 2, DETECTION_SIZE / 2
        half = LOCAL_CROP_SIZE // 2
        x1 = max(0, min(int(round(cx - half)), DETECTION_SIZE - LOCAL_CROP_SIZE))
        y1 = max(0, min(int(round(cy - half)), DETECTION_SIZE - LOCAL_CROP_SIZE))
        local_224 = cv2.resize(detection_image[y1:y1 + LOCAL_CROP_SIZE, x1:x1 + LOCAL_CROP_SIZE], (IMAGE_SIZE, IMAGE_SIZE), interpolation=cv2.INTER_AREA)

        return pseudo_mask, local_224

    def make_guided_view(rgb_224, pseudo_mask_224):
        attenuation = GUIDED_VIEW_BACKGROUND_ATTENUATION + (1.0 - GUIDED_VIEW_BACKGROUND_ATTENUATION) * pseudo_mask_224[:, :, None]
        return (rgb_224.astype(np.float32) * attenuation).clip(0, 255).astype(np.uint8)

    def to_tensor(img):
        return TF.normalize(TF.to_tensor(Image.fromarray(img)), mean=list(VIT_MEAN), std=list(VIT_STD))

    all_logits, all_labels, all_ids = [], [], []
    with torch.no_grad():
        for _, row in tqdm(test_df.iterrows(), total=len(test_df), desc="M12 test inference"):
            image_full = preprocess_p0_full(row["filepath"])
            global_224 = cv2.resize(image_full, (IMAGE_SIZE, IMAGE_SIZE), interpolation=cv2.INTER_AREA)
            pseudo_mask, local_224 = detect_and_crop_and_mask(image_full)
            guided_224 = make_guided_view(global_224, pseudo_mask)

            global_t = to_tensor(global_224).unsqueeze(0).to(device)
            local_t = to_tensor(local_224).unsqueeze(0).to(device)
            guided_t = to_tensor(guided_224).unsqueeze(0).to(device)

            logits = model(global_t, local_t, guided_t)
            all_logits.append(logits.cpu().numpy()[0])
            all_labels.append(int(row["level"]))
            all_ids.append(row["image"])

    return {"logits": np.array(all_logits), "labels": np.array(all_labels), "image_ids": all_ids}


def run_efficientnet_test_inference() -> dict:
    """Genuine deterministic test inference for the EfficientNet-B4 baseline
    (exp01_shared_p0_wrs_focal_seed42), reusing notebook 02's architecture (timm
    efficientnet_b4, native num_classes head) and shared P0 preprocessing. Loads the frozen
    checkpoint only - does not train, does not select an epoch."""
    import cv2
    import timm
    from PIL import Image
    from tqdm.auto import tqdm
    import torchvision.transforms.functional as TF

    EFFICIENTNET_MEAN = (0.485, 0.456, 0.406)
    EFFICIENTNET_STD = (0.229, 0.224, 0.225)
    EFFICIENTNET_IMAGE_SIZE = 224

    checkpoint_path = (
        PROJECT_ROOT / "checkpoints" / "efficientnet_b4"
        / "exp01_shared_p0_wrs_focal_seed42" / "best_model.pt"
    )
    if not checkpoint_path.exists():
        raise FileNotFoundError(f"EfficientNet-B4 checkpoint not found at {checkpoint_path}.")

    model = timm.create_model("efficientnet_b4", pretrained=False, num_classes=NUM_CLASSES).to(device)
    checkpoint = torch.load(checkpoint_path, map_location=device, weights_only=False)
    state_dict = checkpoint["model_state"] if "model_state" in checkpoint else checkpoint
    model.load_state_dict(state_dict)
    model.eval()
    print(f"EfficientNet-B4: loaded frozen checkpoint from {checkpoint_path.name} - test-only inference, no training.")

    def preprocess_p0(image_path):
        img_bgr = cv2.imread(str(image_path))
        img = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)
        gray = cv2.cvtColor(img, cv2.COLOR_RGB2GRAY)
        _, mask = cv2.threshold(gray, 10, 255, cv2.THRESH_BINARY)
        kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (25, 25))
        mask = cv2.morphologyEx(mask, cv2.MORPH_CLOSE, kernel)
        contours, _ = cv2.findContours(mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
        if contours:
            x, y, w, h = cv2.boundingRect(max(contours, key=cv2.contourArea))
            margin = int(min(w, h) * 0.02)
            x, y = max(0, x - margin), max(0, y - margin)
            w = min(img.shape[1] - x, w + 2 * margin)
            h = min(img.shape[0] - y, h + 2 * margin)
            img = img[y:y + h, x:x + w]
        h, w = img.shape[:2]
        side = max(h, w)
        canvas = np.zeros((side, side, 3), dtype=img.dtype)
        canvas[(side - h) // 2:(side - h) // 2 + h, (side - w) // 2:(side - w) // 2 + w] = img
        return cv2.resize(canvas, (EFFICIENTNET_IMAGE_SIZE, EFFICIENTNET_IMAGE_SIZE), interpolation=cv2.INTER_AREA)

    all_logits, all_labels, all_ids = [], [], []
    with torch.no_grad():
        for _, row in tqdm(test_df.iterrows(), total=len(test_df), desc="EfficientNet-B4 test inference"):
            image_224 = preprocess_p0(row["filepath"])
            tensor = TF.normalize(
                TF.to_tensor(Image.fromarray(image_224)), mean=list(EFFICIENTNET_MEAN), std=list(EFFICIENTNET_STD)
            ).unsqueeze(0).to(device)
            logits = model(tensor)
            all_logits.append(logits.cpu().numpy()[0])
            all_labels.append(int(row["level"]))
            all_ids.append(row["image"])

    return {"logits": np.array(all_logits), "labels": np.array(all_labels), "image_ids": all_ids}


def run_deit_test_inference() -> dict:
    """Genuine deterministic test inference for the frozen D07 (two-phase fine-tuned
    DeiT-III) checkpoint, reusing D07's architecture (timm deit3_base_patch16_224.fb_in1k,
    native num_classes head) and P0 preprocessing. Does NOT reconstruct logits from saved
    validation probabilities - this runs genuine forward passes on the test images."""
    import cv2
    import timm
    from PIL import Image
    from tqdm.auto import tqdm
    import torchvision.transforms.functional as TF

    DEIT_MEAN = (0.485, 0.456, 0.406)
    DEIT_STD = (0.229, 0.224, 0.225)
    DEIT_IMAGE_SIZE = 224

    checkpoint_path = PROJECT_ROOT / "checkpoints" / "deit3_b16" / "D07_two_phase_finetuning" / "best.pt"
    if not checkpoint_path.exists():
        raise FileNotFoundError(f"DeiT-III (D07) checkpoint not found at {checkpoint_path}.")

    model = timm.create_model(
        "deit3_base_patch16_224.fb_in1k", pretrained=False, num_classes=NUM_CLASSES
    ).to(device)
    checkpoint = torch.load(checkpoint_path, map_location=device, weights_only=False)
    state_dict = checkpoint["model_state"] if "model_state" in checkpoint else checkpoint
    model.load_state_dict(state_dict)
    model.eval()
    print(f"DeiT-III (D07): loaded frozen checkpoint from epoch {checkpoint.get('epoch', 'n/a')} - test-only inference, no training.")

    def preprocess_p0(image_path):
        img_bgr = cv2.imread(str(image_path))
        img = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)
        gray = cv2.cvtColor(img, cv2.COLOR_RGB2GRAY)
        _, mask = cv2.threshold(gray, 10, 255, cv2.THRESH_BINARY)
        kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (25, 25))
        mask = cv2.morphologyEx(mask, cv2.MORPH_CLOSE, kernel)
        contours, _ = cv2.findContours(mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
        if contours:
            x, y, w, h = cv2.boundingRect(max(contours, key=cv2.contourArea))
            margin = int(min(w, h) * 0.02)
            x, y = max(0, x - margin), max(0, y - margin)
            w = min(img.shape[1] - x, w + 2 * margin)
            h = min(img.shape[0] - y, h + 2 * margin)
            img = img[y:y + h, x:x + w]
        h, w = img.shape[:2]
        side = max(h, w)
        canvas = np.zeros((side, side, 3), dtype=img.dtype)
        canvas[(side - h) // 2:(side - h) // 2 + h, (side - w) // 2:(side - w) // 2 + w] = img
        return cv2.resize(canvas, (DEIT_IMAGE_SIZE, DEIT_IMAGE_SIZE), interpolation=cv2.INTER_AREA)

    all_logits, all_labels, all_ids = [], [], []
    with torch.no_grad():
        for _, row in tqdm(test_df.iterrows(), total=len(test_df), desc="DeiT-III (D07) test inference"):
            image_224 = preprocess_p0(row["filepath"])
            tensor = TF.normalize(
                TF.to_tensor(Image.fromarray(image_224)), mean=list(DEIT_MEAN), std=list(DEIT_STD)
            ).unsqueeze(0).to(device)
            logits = model(tensor)
            all_logits.append(logits.cpu().numpy()[0])
            all_labels.append(int(row["level"]))
            all_ids.append(row["image"])

    return {"logits": np.array(all_logits), "labels": np.array(all_labels), "image_ids": all_ids}


def run_m07_test_inference() -> dict:
    """Genuine deterministic test inference for M07 (single global-view MaxViT-Tiny,
    ordinal-aware loss), reusing M07's own architecture/preprocessing from its config.json."""
    import cv2
    import timm
    from PIL import Image
    from tqdm.auto import tqdm
    import torchvision.transforms.functional as TF

    m07_checkpoint_dir = PROJECT_ROOT / "checkpoints" / "maxvit_tiny" / "M07_ordinal_aware_loss"
    m07_config_path = FROZEN_MODELS["M07"]["log_dir"] / "config.json"
    if not m07_config_path.exists():
        raise FileNotFoundError(f"M07 config not found at {m07_config_path}.")
    with open(m07_config_path) as f:
        m07_config = json.load(f)

    IMAGE_SIZE = m07_config["image_size"]
    VIT_MEAN = tuple(m07_config["normalisation_mean"])
    VIT_STD = tuple(m07_config["normalisation_std"])
    MODEL_NAME = m07_config["model_name"]

    model = timm.create_model(MODEL_NAME, pretrained=False, num_classes=NUM_CLASSES).to(device)
    checkpoint = torch.load(m07_checkpoint_dir / "best.pt", map_location=device, weights_only=False)
    model.load_state_dict(checkpoint["model_state"])
    model.eval()
    print(f"M07: loaded frozen checkpoint from epoch {checkpoint['epoch']} "
          f"(val_qwk={checkpoint['val_qwk']:.4f}) - test-only inference, no training.")

    def preprocess_p0(image_path):
        img_bgr = cv2.imread(str(image_path))
        img = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)
        gray = cv2.cvtColor(img, cv2.COLOR_RGB2GRAY)
        _, mask = cv2.threshold(gray, 10, 255, cv2.THRESH_BINARY)
        kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (25, 25))
        mask = cv2.morphologyEx(mask, cv2.MORPH_CLOSE, kernel)
        contours, _ = cv2.findContours(mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
        if contours:
            x, y, w, h = cv2.boundingRect(max(contours, key=cv2.contourArea))
            margin = int(min(w, h) * 0.02)
            x, y = max(0, x - margin), max(0, y - margin)
            w = min(img.shape[1] - x, w + 2 * margin)
            h = min(img.shape[0] - y, h + 2 * margin)
            img = img[y:y + h, x:x + w]
        h, w = img.shape[:2]
        side = max(h, w)
        canvas = np.zeros((side, side, 3), dtype=img.dtype)
        canvas[(side - h) // 2:(side - h) // 2 + h, (side - w) // 2:(side - w) // 2 + w] = img
        return cv2.resize(canvas, (IMAGE_SIZE, IMAGE_SIZE), interpolation=cv2.INTER_AREA)

    all_logits, all_labels, all_ids = [], [], []
    with torch.no_grad():
        for _, row in tqdm(test_df.iterrows(), total=len(test_df), desc="M07 test inference"):
            image_224 = preprocess_p0(row["filepath"])
            tensor = TF.normalize(
                TF.to_tensor(Image.fromarray(image_224)), mean=list(VIT_MEAN), std=list(VIT_STD)
            ).unsqueeze(0).to(device)
            logits = model(tensor)
            all_logits.append(logits.cpu().numpy()[0])
            all_labels.append(int(row["level"]))
            all_ids.append(row["image"])

    return {"logits": np.array(all_logits), "labels": np.array(all_labels), "image_ids": all_ids}


def run_m11_test_inference() -> dict:
    """Genuine deterministic test inference for M11 (shared-backbone global-local crop
    fusion), reusing M11's own architecture/preprocessing/crop-selection from its config.json."""
    import cv2
    import timm
    from PIL import Image
    from tqdm.auto import tqdm
    import torchvision.transforms.functional as TF

    m11_checkpoint_dir = PROJECT_ROOT / "checkpoints" / "maxvit_tiny" / "M11_global_local_crop_fusion"
    m11_config_path = FROZEN_MODELS["M11"]["log_dir"] / "config.json"
    if not m11_config_path.exists():
        raise FileNotFoundError(f"M11 config not found at {m11_config_path}.")
    with open(m11_config_path) as f:
        m11_config = json.load(f)

    IMAGE_SIZE = m11_config["image_size"]
    VIT_MEAN = tuple(m11_config["normalisation_mean"])
    VIT_STD = tuple(m11_config["normalisation_std"])
    MODEL_NAME = m11_config["model_name"]
    local_crop_cfg = m11_config["local_crop_pipeline"]

    class GlobalLocalMaxViT(nn.Module):
        def __init__(self, model_name, num_classes):
            super().__init__()
            self.backbone = timm.create_model(model_name, pretrained=False, num_classes=0, global_pool="avg")
            feature_dim = self.backbone.num_features
            self.classifier = nn.Linear(feature_dim * 2, num_classes)

        def forward(self, global_image, local_image):
            combined = torch.cat([global_image, local_image], dim=0)
            features = self.backbone(combined)
            g, l = features.chunk(2, dim=0)
            return self.classifier(torch.cat([g, l], dim=1))

    model = GlobalLocalMaxViT(MODEL_NAME, NUM_CLASSES).to(device)
    checkpoint = torch.load(m11_checkpoint_dir / "best.pt", map_location=device, weights_only=False)
    model.load_state_dict(checkpoint["model_state"])
    model.eval()
    print(f"M11: loaded frozen checkpoint from epoch {checkpoint['epoch']} "
          f"(val_qwk={checkpoint['val_qwk']:.4f}) - test-only inference, no training.")

    DETECTION_SIZE = local_crop_cfg["detection_size"]
    CLAHE_CLIP_LIMIT = local_crop_cfg["clahe_clip_limit"]
    CLAHE_TILE_GRID_SIZE = tuple(local_crop_cfg["clahe_tile_grid_size"])
    BACKGROUND_SIGMA = local_crop_cfg["background_sigma"]
    DARK_RESPONSE_PERCENTILE = local_crop_cfg["dark_response_percentile"]
    MIN_COMPONENT_AREA = local_crop_cfg["min_component_area"]
    MAX_COMPONENT_AREA = local_crop_cfg["max_component_area"]
    MAX_ASPECT_RATIO = local_crop_cfg["max_aspect_ratio"]
    MAX_CANDIDATE_COMPONENTS = local_crop_cfg["max_candidate_components"]
    LOCAL_CROP_SIZE = local_crop_cfg["local_crop_size"]

    def preprocess_p0_full(image_path):
        img_bgr = cv2.imread(str(image_path))
        img = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)
        gray = cv2.cvtColor(img, cv2.COLOR_RGB2GRAY)
        _, mask = cv2.threshold(gray, 10, 255, cv2.THRESH_BINARY)
        kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (25, 25))
        mask = cv2.morphologyEx(mask, cv2.MORPH_CLOSE, kernel)
        contours, _ = cv2.findContours(mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
        if contours:
            x, y, w, h = cv2.boundingRect(max(contours, key=cv2.contourArea))
            margin = int(min(w, h) * 0.02)
            x, y = max(0, x - margin), max(0, y - margin)
            w = min(img.shape[1] - x, w + 2 * margin)
            h = min(img.shape[0] - y, h + 2 * margin)
            img = img[y:y + h, x:x + w]
        h, w = img.shape[:2]
        side = max(h, w)
        canvas = np.zeros((side, side, 3), dtype=img.dtype)
        canvas[(side - h) // 2:(side - h) // 2 + h, (side - w) // 2:(side - w) // 2 + w] = img
        return canvas

    def select_local_crop_224(image_full):
        detection_image = cv2.resize(image_full, (DETECTION_SIZE, DETECTION_SIZE), interpolation=cv2.INTER_AREA)
        green = detection_image[:, :, 1]
        clahe = cv2.createCLAHE(clipLimit=CLAHE_CLIP_LIMIT, tileGridSize=CLAHE_TILE_GRID_SIZE)
        enhanced_green = clahe.apply(green)
        background = cv2.GaussianBlur(enhanced_green, (0, 0), BACKGROUND_SIGMA)
        dark_response = cv2.subtract(background, enhanced_green)
        gray = cv2.cvtColor(detection_image, cv2.COLOR_RGB2GRAY)
        field_mask = (gray > 10).astype(np.uint8)
        dark_response = dark_response * field_mask
        field_pixels = dark_response[field_mask > 0]
        if field_pixels.size == 0 or field_pixels.max() <= 0:
            binary = np.zeros_like(field_mask, dtype=np.uint8)
        else:
            threshold_value = float(np.percentile(field_pixels, DARK_RESPONSE_PERCENTILE))
            binary = ((dark_response >= threshold_value) & (dark_response > 0) & (field_mask > 0)).astype(np.uint8) * 255
        kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (3, 3))
        binary = cv2.morphologyEx(binary, cv2.MORPH_OPEN, kernel)
        num_labels, labels_arr, stats, centroids = cv2.connectedComponentsWithStats(binary, connectivity=8)

        candidates = []
        for label_id in range(1, num_labels):
            area = stats[label_id, cv2.CC_STAT_AREA]
            w, h = stats[label_id, cv2.CC_STAT_WIDTH], stats[label_id, cv2.CC_STAT_HEIGHT]
            if area < MIN_COMPONENT_AREA or area > MAX_COMPONENT_AREA:
                continue
            if max(w, h) / max(1, min(w, h)) > MAX_ASPECT_RATIO:
                continue
            score = float(dark_response[labels_arr == label_id].mean())
            cx, cy = centroids[label_id]
            candidates.append({"score": score, "cx": cx, "cy": cy})
        candidates.sort(key=lambda c: c["score"], reverse=True)

        if candidates:
            cx, cy = candidates[0]["cx"], candidates[0]["cy"]
        else:
            cx, cy = DETECTION_SIZE / 2, DETECTION_SIZE / 2
        half = LOCAL_CROP_SIZE // 2
        x1 = max(0, min(int(round(cx - half)), DETECTION_SIZE - LOCAL_CROP_SIZE))
        y1 = max(0, min(int(round(cy - half)), DETECTION_SIZE - LOCAL_CROP_SIZE))
        return cv2.resize(
            detection_image[y1:y1 + LOCAL_CROP_SIZE, x1:x1 + LOCAL_CROP_SIZE],
            (IMAGE_SIZE, IMAGE_SIZE), interpolation=cv2.INTER_AREA,
        )

    def to_tensor(img):
        return TF.normalize(TF.to_tensor(Image.fromarray(img)), mean=list(VIT_MEAN), std=list(VIT_STD))

    all_logits, all_labels, all_ids = [], [], []
    with torch.no_grad():
        for _, row in tqdm(test_df.iterrows(), total=len(test_df), desc="M11 test inference"):
            image_full = preprocess_p0_full(row["filepath"])
            global_224 = cv2.resize(image_full, (IMAGE_SIZE, IMAGE_SIZE), interpolation=cv2.INTER_AREA)
            local_224 = select_local_crop_224(image_full)

            global_t = to_tensor(global_224).unsqueeze(0).to(device)
            local_t = to_tensor(local_224).unsqueeze(0).to(device)

            logits = model(global_t, local_t)
            all_logits.append(logits.cpu().numpy()[0])
            all_labels.append(int(row["level"]))
            all_ids.append(row["image"])

    return {"logits": np.array(all_logits), "labels": np.array(all_labels), "image_ids": all_ids}


print("Inference functions ready: EfficientNet-B4, DeiT-III, M07, M11, M12 all implemented "
      "with genuine architecture/checkpoint loading and deterministic test-only inference.")

In [ ]:
test_predictions_by_model = {}
for model_name, inference_fn in [
    ("EfficientNet-B4", run_efficientnet_test_inference),
    ("DeiT-III", run_deit_test_inference),
    ("M12", run_m12_test_inference),
]:
    test_predictions_by_model[model_name] = get_or_run_test_inference(
        model_name, FROZEN_MODELS[model_name]["log_dir"], inference_fn
    )
    print(f"{model_name}: {len(test_predictions_by_model[model_name]['image_ids']):,} test predictions obtained.")

In [ ]:
for model_name in ["EfficientNet-B4", "DeiT-III", "M12"]:
    data = test_predictions_by_model[model_name]
    frozen_dir = FROZEN_MODELS[model_name]["log_dir"] / "A01_frozen_test_predictions"

    pred_df = pd.DataFrame({"image_id": data["image_ids"], "true_label": data["labels"]})
    pred_df["predicted_label"] = data["probs"].argmax(axis=1)
    pred_df["confidence"] = data["probs"].max(axis=1)
    for c in range(NUM_CLASSES):
        pred_df[f"logit_grade_{c}"] = data["logits"][:, c]
    for c in range(NUM_CLASSES):
        pred_df[f"prob_grade_{c}"] = data["probs"][:, c]

    np.save(frozen_dir / "test_probabilities.npy", data["probs"])
    pred_df.to_csv(frozen_dir / "test_predictions.csv", index=False)
    print(f"{model_name}: saved test_probabilities.npy and test_predictions.csv -> {frozen_dir}")

## 6. Apply the frozen M13 ensemble

Uses **exactly** the saved M13 configuration — `model_temperatures.json`,
`class_specialist_weights.csv`, `final_ensemble_config.json`. **No refitting of temperatures
or weights happens here.** M13-D's constituent models are whichever models the frozen M13
notebook actually selected (`final_ensemble_config.json["models_included"]`) — this can
include EfficientNet-B4, DeiT-III, M07, M09, M10, M11, M12 depending on what M13 chose;
whichever subset it is, this section obtains **frozen test predictions for exactly that same
set** before combining them, extending Section 5's inference machinery to any constituent
model not already covered there.

In [ ]:
M13_LOG_DIR = FROZEN_MODELS["M13-D"]["log_dir"]

with open(M13_LOG_DIR / "final_ensemble_config.json") as f:
    m13_final_config = json.load(f)
with open(M13_LOG_DIR / "model_temperatures.json") as f:
    m13_temperatures = json.load(f)
m13_class_weights_df = pd.read_csv(M13_LOG_DIR / "class_specialist_weights.csv", index_col=0)

m13_constituent_models = m13_final_config["models_included"]
assert m13_final_config["selected_variant"] == "M13-D", (
    f"final_ensemble_config.json selected {m13_final_config['selected_variant']}, not M13-D - "
    "the A01 brief specifies M13-D as the final ensemble; if a different variant was actually "
    "selected, update the brief/this cell rather than silently substituting one for the other."
)

print(f"M13-D constituent models: {m13_constituent_models}")
print(f"M13-D temperatures: {m13_temperatures}")
print("Class-specialist weights:")
print(m13_class_weights_df)

In [ ]:
# Extend Section 5's inference machinery to cover any M13 constituent not already obtained.
inference_fn_by_model = {
    "EfficientNet-B4": run_efficientnet_test_inference,
    "DeiT-III": run_deit_test_inference,
    "M07": run_m07_test_inference,
    "M11": run_m11_test_inference,
    "M12": run_m12_test_inference,
}

for model_name in m13_constituent_models:
    if model_name in test_predictions_by_model:
        continue
    if model_name not in FROZEN_MODELS:
        raise KeyError(
            f"M13-D requires '{model_name}', which is not registered in FROZEN_MODELS - "
            "add its log_dir/architecture details before proceeding."
        )
    if model_name not in inference_fn_by_model:
        raise NotImplementedError(
            f"M13-D requires test predictions for '{model_name}', but no inference function is "
            "registered for it in this notebook (only EfficientNet-B4/DeiT-III/M07/M11/M12 are). "
            f"Add a run_{model_name.lower()}_test_inference() function using that model's own "
            "architecture/checkpoint code before this cell can proceed."
        )
    test_predictions_by_model[model_name] = get_or_run_test_inference(
        model_name, FROZEN_MODELS[model_name]["log_dir"], inference_fn_by_model[model_name]
    )
    print(f"{model_name}: obtained test predictions for M13-D constituent use.")

print(f"\nAll M13-D constituents available: {m13_constituent_models}")

In [ ]:
# Align all constituent models by image_id and verify labels match exactly.
constituent_id_sets = {name: set(test_predictions_by_model[name]["image_ids"]) for name in m13_constituent_models}
assert all(ids == set(REFERENCE_TEST_IMAGE_IDS) for ids in constituent_id_sets.values()), (
    "Not every M13-D constituent model covers the full reference test image ID set."
)

aligned_test_ids = REFERENCE_TEST_IMAGE_IDS
aligned_test_labels = np.array([REFERENCE_TEST_LABELS[image_id] for image_id in aligned_test_ids])

calibrated_test_probs = {}
for model_name in m13_constituent_models:
    data = test_predictions_by_model[model_name]
    id_to_row = {image_id: i for i, image_id in enumerate(data["image_ids"])}
    row_order = [id_to_row[image_id] for image_id in aligned_test_ids]
    model_labels = np.array(data["labels"])[row_order]
    assert np.array_equal(model_labels, aligned_test_labels), f"{model_name}'s test labels do not match the reference after alignment."

    logits_ordered = data["logits"][row_order]
    temperature = m13_temperatures[model_name]
    calibrated_logits = logits_ordered / temperature
    calibrated_probs = F.softmax(torch.tensor(calibrated_logits, dtype=torch.float64), dim=1).numpy()

    assert np.isfinite(calibrated_probs).all(), f"{model_name} produced non-finite calibrated probabilities."
    assert np.allclose(calibrated_probs.sum(axis=1), 1.0, atol=1e-6), f"{model_name}'s calibrated probabilities do not sum to 1."

    calibrated_test_probs[model_name] = calibrated_probs

print("assert identical image IDs   : PASSED")
print("assert identical labels      : PASSED")
print("assert probabilities finite  : PASSED")
print("assert probabilities sum ~1  : PASSED")
print("assert no test-set fitting occurred : PASSED (temperatures/weights loaded from frozen M13 files, never refit here)")

In [ ]:
class_weight_matrix = m13_class_weights_df.loc[m13_constituent_models, CLASS_NAMES].to_numpy()

combined_score = np.zeros((len(aligned_test_ids), NUM_CLASSES))
for class_id in range(NUM_CLASSES):
    for model_idx, model_name in enumerate(m13_constituent_models):
        combined_score[:, class_id] += class_weight_matrix[model_idx, class_id] * calibrated_test_probs[model_name][:, class_id]

row_sums = combined_score.sum(axis=1, keepdims=True)
row_sums[row_sums == 0] = 1.0
m13_test_probs = combined_score / row_sums
m13_test_preds = m13_test_probs.argmax(axis=1)

assert np.isfinite(m13_test_probs).all()
assert np.allclose(m13_test_probs.sum(axis=1), 1.0, atol=1e-6)

np.save(A01_LOG_DIR / "m13_test_probabilities.npy", m13_test_probs)

m13_test_predictions_df = pd.DataFrame({
    "image_id": aligned_test_ids, "true_label": aligned_test_labels, "predicted_label": m13_test_preds,
    **{f"prob_{c}": m13_test_probs[:, c] for c in range(NUM_CLASSES)},
})
m13_test_predictions_df.to_csv(A01_LOG_DIR / "m13_test_predictions.csv", index=False)

print(f"Saved -> {A01_LOG_DIR / 'm13_test_probabilities.npy'}")
print(f"Saved -> {A01_LOG_DIR / 'm13_test_predictions.csv'}")
print(f"\nM13-D test QWK (frozen system, no refitting): "
      f"{cohen_kappa_score(aligned_test_labels, m13_test_preds, weights='quadratic'):.4f}")

## 7. Overall metrics

Calculated for every final model (EfficientNet-B4, DeiT-III, M12, M13-D). QWK is the primary
ranking metric.

In [ ]:
def compute_overall_metrics(labels: np.ndarray, probs: np.ndarray) -> dict:
    preds = probs.argmax(axis=1)
    absolute_error = np.abs(preds - labels)
    return {
        "qwk": cohen_kappa_score(labels, preds, weights="quadratic"),
        "macro_f1": f1_score(labels, preds, average="macro", zero_division=0),
        "weighted_f1": f1_score(labels, preds, average="weighted", zero_division=0),
        "balanced_accuracy": balanced_accuracy_score(labels, preds),
        "accuracy": accuracy_score(labels, preds),
        "macro_precision": precision_score(labels, preds, average="macro", zero_division=0),
        "macro_recall": recall_score(labels, preds, average="macro", zero_division=0),
        "nll": float(log_loss(labels, probs, labels=list(range(NUM_CLASSES)))),
        "brier": float(np.mean(np.sum((probs - np.eye(NUM_CLASSES)[labels]) ** 2, axis=1))),
        "mae_grade": float(absolute_error.mean()),
        "large_grade_error_rate": float((absolute_error >= 2).mean()),
    }


def expected_calibration_error(labels: np.ndarray, probs: np.ndarray, n_bins: int = 15) -> float:
    confidences = probs.max(axis=1)
    predictions = probs.argmax(axis=1)
    correctness = (predictions == labels).astype(float)
    bin_edges = np.linspace(0.0, 1.0, n_bins + 1)
    ece = 0.0
    for bin_idx in range(n_bins):
        lo, hi = bin_edges[bin_idx], bin_edges[bin_idx + 1]
        in_bin = (confidences > lo) & (confidences <= hi) if bin_idx > 0 else (confidences >= lo) & (confidences <= hi)
        if in_bin.sum() == 0:
            continue
        gap = abs(confidences[in_bin].mean() - correctness[in_bin].mean())
        ece += (in_bin.sum() / len(confidences)) * gap
    return float(ece)


final_test_probs = {
    "EfficientNet-B4": test_predictions_by_model["EfficientNet-B4"]["probs"],
    "DeiT-III": test_predictions_by_model["DeiT-III"]["probs"],
    "M12": test_predictions_by_model["M12"]["probs"],
    "M13-D": m13_test_probs,
}
FINAL_MODELS = ["EfficientNet-B4", "DeiT-III", "M12", "M13-D"]

# Re-align every model's probs to aligned_test_ids order (Section 5's models weren't
# necessarily aligned yet; M13-D already is).
for model_name in ["EfficientNet-B4", "DeiT-III", "M12"]:
    data = test_predictions_by_model[model_name]
    id_to_row = {image_id: i for i, image_id in enumerate(data["image_ids"])}
    row_order = [id_to_row[image_id] for image_id in aligned_test_ids]
    final_test_probs[model_name] = data["probs"][row_order]

overall_metrics_rows = []
for model_name in FINAL_MODELS:
    metrics = compute_overall_metrics(aligned_test_labels, final_test_probs[model_name])
    metrics["ece"] = expected_calibration_error(aligned_test_labels, final_test_probs[model_name])
    metrics["model"] = model_name
    overall_metrics_rows.append(metrics)

final_test_metrics_df = pd.DataFrame(overall_metrics_rows).set_index("model")
final_test_metrics_df = final_test_metrics_df[[
    "qwk", "macro_f1", "weighted_f1", "balanced_accuracy", "accuracy",
    "macro_precision", "macro_recall", "nll", "brier", "ece", "mae_grade", "large_grade_error_rate",
]]
print(final_test_metrics_df.round(4).sort_values("qwk", ascending=False).to_string())

final_test_metrics_df.reset_index().to_csv(A01_LOG_DIR / "final_test_metrics.csv", index=False)
print(f"\nSaved -> {A01_LOG_DIR / 'final_test_metrics.csv'}")

## 8. Per-class metrics

Particular attention to Mild, Severe, and Proliferative DR — the classes most likely to
matter clinically and be underrepresented.

In [ ]:
per_class_rows = []
for model_name in FINAL_MODELS:
    probs = final_test_probs[model_name]
    preds = probs.argmax(axis=1)
    precisions = precision_score(aligned_test_labels, preds, average=None, labels=list(range(NUM_CLASSES)), zero_division=0)
    recalls = recall_score(aligned_test_labels, preds, average=None, labels=list(range(NUM_CLASSES)), zero_division=0)
    f1s = f1_score(aligned_test_labels, preds, average=None, labels=list(range(NUM_CLASSES)), zero_division=0)
    supports = np.bincount(aligned_test_labels, minlength=NUM_CLASSES)
    for class_id in range(NUM_CLASSES):
        binary_labels = (aligned_test_labels == class_id).astype(int)
        ap = average_precision_score(binary_labels, probs[:, class_id])
        per_class_rows.append({
            "model": model_name, "class": CLASS_NAMES[class_id],
            "precision": precisions[class_id], "recall": recalls[class_id], "f1": f1s[class_id],
            "support": int(supports[class_id]), "average_precision": ap,
        })

final_test_per_class_metrics_df = pd.DataFrame(per_class_rows)
print(final_test_per_class_metrics_df.round(4).to_string(index=False))

final_test_per_class_metrics_df.to_csv(A01_LOG_DIR / "final_test_per_class_metrics.csv", index=False)
print(f"\nSaved -> {A01_LOG_DIR / 'final_test_per_class_metrics.csv'}")

print("\nMild / Severe / PDR summary:")
for class_name in ["Mild", "Severe", "Proliferative DR"]:
    sub = final_test_per_class_metrics_df[final_test_per_class_metrics_df["class"] == class_name]
    print(f"  {class_name}:")
    print(sub[["model", "precision", "recall", "f1"]].to_string(index=False))

## 9. Confusion matrices

One raw-count and one row-normalised confusion matrix per final model, plus a grade-distance
distribution (errors of 0/1/2/3/4 grades) across all four.

In [ ]:
figure_filenames = {
    "EfficientNet-B4": "efficientnet_test_confusion_matrix.png",
    "DeiT-III": "deit_test_confusion_matrix.png",
    "M12": "m12_test_confusion_matrix.png",
    "M13-D": "m13_test_confusion_matrix.png",
}
for model_name in FINAL_MODELS:
    preds = final_test_probs[model_name].argmax(axis=1)
    cm_counts = confusion_matrix(aligned_test_labels, preds)
    cm_normalised = cm_counts.astype(float) / cm_counts.sum(axis=1, keepdims=True)

    fig, axes = plt.subplots(1, 2, figsize=(11, 4.8))
    for ax, data, fmt, title in [
        (axes[0], cm_counts, "d", f"{model_name} -- counts"),
        (axes[1], cm_normalised, ".2f", f"{model_name} -- normalised"),
    ]:
        im = ax.imshow(data, cmap="Purples")
        ax.set_xticks(range(NUM_CLASSES)); ax.set_yticks(range(NUM_CLASSES))
        ax.set_xticklabels(CLASS_NAMES, rotation=45, ha="right"); ax.set_yticklabels(CLASS_NAMES)
        ax.set_xlabel("Predicted"); ax.set_ylabel("True")
        ax.set_title(title, fontweight="bold")
        thresh = data.max() / 2
        for i in range(NUM_CLASSES):
            for j in range(NUM_CLASSES):
                ax.text(j, i, format(data[i, j], fmt), ha="center", va="center", color="white" if data[i, j] > thresh else "black")
        fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
    plt.tight_layout()
    plt.savefig(A01_FIGURE_DIR / figure_filenames[model_name], dpi=140, bbox_inches="tight")
    plt.show()
    print(f"Saved -> {A01_FIGURE_DIR / figure_filenames[model_name]}")

In [ ]:
ordinal_distance_data = {}
for model_name in FINAL_MODELS:
    preds = final_test_probs[model_name].argmax(axis=1)
    distances = np.abs(preds - aligned_test_labels)
    ordinal_distance_data[model_name] = [int(np.sum(distances == d)) for d in range(5)]

ordinal_distance_df = pd.DataFrame(ordinal_distance_data, index=[f"{d} grade(s)" for d in range(5)])
print(ordinal_distance_df)

fig, ax = plt.subplots(figsize=(9, 5.5))
ordinal_distance_df.plot(kind="bar", ax=ax)
ax.set_ylabel("Number of test images")
ax.set_title("Grade-distance error distribution -- final models", fontweight="bold")
plt.xticks(rotation=0)
plt.tight_layout()
plt.savefig(A01_FIGURE_DIR / "ordinal_error_distance_comparison.png", dpi=140, bbox_inches="tight")
plt.show()
print(f"Saved -> {A01_FIGURE_DIR / 'ordinal_error_distance_comparison.png'}")

## 10. Bootstrap confidence intervals

**Patient-level** bootstrap resampling (patient IDs are available in the test split) — each
bootstrap sample draws patients with replacement, then includes all of that patient's test
images, respecting the patient-grouped structure of the original split rather than treating
individual images as independent. 2,000 resamples, fixed seed, 95% percentile CIs.

In [ ]:
test_patient_ids = np.array([
    test_df.set_index("image").loc[image_id, "patient_id"] for image_id in aligned_test_ids
])
unique_patients = np.unique(test_patient_ids)
patient_to_indices = {p: np.where(test_patient_ids == p)[0] for p in unique_patients}

rng = np.random.default_rng(BOOTSTRAP_SEED)

def bootstrap_metric_samples(labels: np.ndarray, preds: np.ndarray, metric_fn, n_bootstrap: int = N_BOOTSTRAP) -> np.ndarray:
    samples = np.empty(n_bootstrap)
    for b in range(n_bootstrap):
        sampled_patients = rng.choice(unique_patients, size=len(unique_patients), replace=True)
        indices = np.concatenate([patient_to_indices[p] for p in sampled_patients])
        samples[b] = metric_fn(labels[indices], preds[indices])
    return samples


bootstrap_metric_fns = {
    "qwk": lambda y, p: cohen_kappa_score(y, p, weights="quadratic"),
    "macro_f1": lambda y, p: f1_score(y, p, average="macro", zero_division=0),
    "balanced_accuracy": lambda y, p: balanced_accuracy_score(y, p),
    "accuracy": lambda y, p: accuracy_score(y, p),
    "mae_grade": lambda y, p: float(np.mean(np.abs(p - y))),
}

bootstrap_ci_rows = []
bootstrap_samples_by_model_metric = {}
for model_name in FINAL_MODELS:
    preds = final_test_probs[model_name].argmax(axis=1)
    for metric_name, metric_fn in bootstrap_metric_fns.items():
        samples = bootstrap_metric_samples(aligned_test_labels, preds, metric_fn)
        bootstrap_samples_by_model_metric[(model_name, metric_name)] = samples
        estimate = metric_fn(aligned_test_labels, preds)
        ci_lower, ci_upper = np.percentile(samples, CI_PERCENTILES)
        bootstrap_ci_rows.append({
            "model": model_name, "metric": metric_name, "estimate": estimate,
            "ci_lower": ci_lower, "ci_upper": ci_upper,
            "bootstrap_unit": "patient", "n_bootstrap": N_BOOTSTRAP,
        })
    print(f"{model_name}: bootstrap CIs computed.")

bootstrap_confidence_intervals_df = pd.DataFrame(bootstrap_ci_rows)
print(bootstrap_confidence_intervals_df.round(4).to_string(index=False))

bootstrap_confidence_intervals_df.to_csv(A01_LOG_DIR / "bootstrap_confidence_intervals.csv", index=False)
print(f"\nSaved -> {A01_LOG_DIR / 'bootstrap_confidence_intervals.csv'}")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5.5))
for ax, metric_name in zip(axes, ["qwk", "macro_f1"]):
    sub = bootstrap_confidence_intervals_df[bootstrap_confidence_intervals_df["metric"] == metric_name]
    ax.bar(sub["model"], sub["estimate"], yerr=[sub["estimate"] - sub["ci_lower"], sub["ci_upper"] - sub["estimate"]], capsize=5, color="#3498DB")
    ax.set_title(f"{metric_name.upper()} with 95% patient-bootstrap CI", fontweight="bold")
    ax.set_ylabel(metric_name.upper())
plt.xticks(rotation=20, ha="right")
plt.tight_layout()
plt.savefig(A01_FIGURE_DIR / "qwk_macro_f1_with_confidence_intervals.png", dpi=140, bbox_inches="tight")
plt.show()
print(f"Saved -> {A01_FIGURE_DIR / 'qwk_macro_f1_with_confidence_intervals.png'}")

## 11. Paired model comparisons

Paired bootstrap differences (same resampled patients feed both models in each pair, since
every model predicts the identical test samples). A confidence interval that excludes zero is
required before describing a difference as supported by the data — point-estimate differences
alone are not treated as evidence of superiority.

In [ ]:
PAIRED_COMPARISONS = [
    ("M13-D", "M12"), ("M13-D", "DeiT-III"), ("M13-D", "EfficientNet-B4"),
    ("M12", "DeiT-III"), ("M12", "EfficientNet-B4"),
]

paired_bootstrap_rows = []
for model_a, model_b in PAIRED_COMPARISONS:
    preds_a = final_test_probs[model_a].argmax(axis=1)
    preds_b = final_test_probs[model_b].argmax(axis=1)

    for metric_name, metric_fn in [
        ("qwk", bootstrap_metric_fns["qwk"]), ("macro_f1", bootstrap_metric_fns["macro_f1"]),
        ("balanced_accuracy", bootstrap_metric_fns["balanced_accuracy"]),
    ]:
        diffs = np.empty(N_BOOTSTRAP)
        rng_pair = np.random.default_rng(BOOTSTRAP_SEED)
        for b in range(N_BOOTSTRAP):
            sampled_patients = rng_pair.choice(unique_patients, size=len(unique_patients), replace=True)
            indices = np.concatenate([patient_to_indices[p] for p in sampled_patients])
            diffs[b] = metric_fn(aligned_test_labels[indices], preds_a[indices]) - metric_fn(aligned_test_labels[indices], preds_b[indices])

        point_diff = metric_fn(aligned_test_labels, preds_a) - metric_fn(aligned_test_labels, preds_b)
        ci_lower, ci_upper = np.percentile(diffs, CI_PERCENTILES)
        prop_a_exceeds_b = float((diffs > 0).mean())

        paired_bootstrap_rows.append({
            "model_a": model_a, "model_b": model_b, "metric": metric_name,
            "difference": point_diff, "ci_lower": ci_lower, "ci_upper": ci_upper,
            "ci_excludes_zero": bool(ci_lower > 0 or ci_upper < 0),
            "proportion_a_exceeds_b": prop_a_exceeds_b,
        })

paired_bootstrap_model_differences_df = pd.DataFrame(paired_bootstrap_rows)
print(paired_bootstrap_model_differences_df.round(4).to_string(index=False))

paired_bootstrap_model_differences_df.to_csv(A01_LOG_DIR / "paired_bootstrap_model_differences.csv", index=False)
print(f"\nSaved -> {A01_LOG_DIR / 'paired_bootstrap_model_differences.csv'}")

print("\nComparisons with a QWK confidence interval excluding zero (i.e. a supported difference):")
supported = paired_bootstrap_model_differences_df[
    (paired_bootstrap_model_differences_df["metric"] == "qwk") & (paired_bootstrap_model_differences_df["ci_excludes_zero"])
]
print(supported[["model_a", "model_b", "difference", "ci_lower", "ci_upper"]].to_string(index=False) if len(supported) > 0 else "  (none)")

## 12. Prediction disagreement analysis

Especially: M13-D vs M12, M12 vs DeiT-III, M12 vs EfficientNet-B4.

In [ ]:
model_preds_final = {name: final_test_probs[name].argmax(axis=1) for name in FINAL_MODELS}
model_correct_final = {name: (model_preds_final[name] == aligned_test_labels) for name in FINAL_MODELS}
model_error_sets_final = {name: set(np.array(aligned_test_ids)[~model_correct_final[name]]) for name in FINAL_MODELS}

complementarity_rows = []
for model_a, model_b in combinations(FINAL_MODELS, 2):
    both_correct = int(np.sum(model_correct_final[model_a] & model_correct_final[model_b]))
    both_wrong = int(np.sum(~model_correct_final[model_a] & ~model_correct_final[model_b]))
    a_correct_b_wrong = int(np.sum(model_correct_final[model_a] & ~model_correct_final[model_b]))
    a_wrong_b_correct = int(np.sum(~model_correct_final[model_a] & model_correct_final[model_b]))
    agreement_rate = float(np.mean(model_preds_final[model_a] == model_preds_final[model_b]))
    kappa = cohen_kappa_score(model_preds_final[model_a], model_preds_final[model_b])
    error_a, error_b = model_error_sets_final[model_a], model_error_sets_final[model_b]
    union = error_a | error_b
    jaccard = len(error_a & error_b) / len(union) if len(union) > 0 else float("nan")

    complementarity_rows.append({
        "model_a": model_a, "model_b": model_b, "both_correct": both_correct, "both_wrong": both_wrong,
        "a_correct_b_wrong": a_correct_b_wrong, "a_wrong_b_correct": a_wrong_b_correct,
        "agreement_rate": agreement_rate, "cohens_kappa": kappa, "error_jaccard_similarity": jaccard,
    })

final_model_error_complementarity_df = pd.DataFrame(complementarity_rows)
print(final_model_error_complementarity_df.round(4).to_string(index=False))
final_model_error_complementarity_df.to_csv(A01_LOG_DIR / "final_model_error_complementarity.csv", index=False)
print(f"\nSaved -> {A01_LOG_DIR / 'final_model_error_complementarity.csv'}")

In [ ]:
transition_rows = []
for model_a, model_b in [("M13-D", "M12"), ("M12", "DeiT-III"), ("M12", "EfficientNet-B4")]:
    corrected = int(np.sum(model_correct_final[model_a] & ~model_correct_final[model_b]))
    newly_wrong = int(np.sum(~model_correct_final[model_a] & model_correct_final[model_b]))
    jointly_wrong = int(np.sum(~model_correct_final[model_a] & ~model_correct_final[model_b]))
    transition_rows.append({
        "model_a": model_a, "model_b": model_b,
        "a_corrects_relative_to_b": corrected, "a_newly_wrong_relative_to_b": newly_wrong,
        "jointly_wrong": jointly_wrong,
    })

final_model_error_transitions_df = pd.DataFrame(transition_rows)
print(final_model_error_transitions_df.to_string(index=False))
final_model_error_transitions_df.to_csv(A01_LOG_DIR / "final_model_error_transitions.csv", index=False)
print(f"\nSaved -> {A01_LOG_DIR / 'final_model_error_transitions.csv'}")

fig, ax = plt.subplots(figsize=(8, 5.5))
m13_vs_m12_row = final_model_error_transitions_df[
    (final_model_error_transitions_df["model_a"] == "M13-D") & (final_model_error_transitions_df["model_b"] == "M12")
].iloc[0]
ax.bar(
    ["M13-D corrects M12", "M13-D newly wrong", "Both wrong"],
    [m13_vs_m12_row["a_corrects_relative_to_b"], m13_vs_m12_row["a_newly_wrong_relative_to_b"], m13_vs_m12_row["jointly_wrong"]],
    color=["#2ECC71", "#E74C3C", "#95A5A6"],
)
ax.set_ylabel("Count")
ax.set_title("M13-D vs M12 -- error transitions", fontweight="bold")
plt.tight_layout()
plt.savefig(A01_FIGURE_DIR / "m13_vs_m12_error_transitions.png", dpi=140, bbox_inches="tight")
plt.show()
print(f"Saved -> {A01_FIGURE_DIR / 'm13_vs_m12_error_transitions.png'}")

## 13. Calibration analysis

Important because M13 specifically includes calibration — this section checks whether it
actually delivered better-calibrated confidence on the test set, not just on validation.

In [ ]:
calibration_rows = []
for model_name in FINAL_MODELS:
    probs = final_test_probs[model_name]
    preds = probs.argmax(axis=1)
    confidences = probs.max(axis=1)
    correct_mask = preds == aligned_test_labels

    calibration_rows.append({
        "model": model_name,
        "nll": float(log_loss(aligned_test_labels, probs, labels=list(range(NUM_CLASSES)))),
        "brier": float(np.mean(np.sum((probs - np.eye(NUM_CLASSES)[aligned_test_labels]) ** 2, axis=1))),
        "ece": expected_calibration_error(aligned_test_labels, probs),
        "mean_confidence_correct": float(confidences[correct_mask].mean()) if correct_mask.sum() > 0 else float("nan"),
        "mean_confidence_wrong": float(confidences[~correct_mask].mean()) if (~correct_mask).sum() > 0 else float("nan"),
        "n_high_confidence_wrong_ge_0.80": int(np.sum((confidences >= 0.80) & ~correct_mask)),
    })
calibration_df = pd.DataFrame(calibration_rows)
print(calibration_df.round(4).to_string(index=False))

In [ ]:
fig, axes = plt.subplots(1, len(FINAL_MODELS), figsize=(5 * len(FINAL_MODELS), 4.5))
for ax, model_name in zip(axes, FINAL_MODELS):
    probs = final_test_probs[model_name]
    confidences = probs.max(axis=1)
    preds = probs.argmax(axis=1)
    correctness = (preds == aligned_test_labels).astype(float)
    bin_edges = np.linspace(0, 1, 16)
    bin_acc, bin_conf = [], []
    for i in range(15):
        lo, hi = bin_edges[i], bin_edges[i + 1]
        in_bin = (confidences > lo) & (confidences <= hi) if i > 0 else (confidences >= lo) & (confidences <= hi)
        if in_bin.sum() > 0:
            bin_acc.append(correctness[in_bin].mean())
            bin_conf.append(bin_edges[i])
        else:
            bin_acc.append(np.nan)
            bin_conf.append(bin_edges[i])
    ax.bar(bin_conf, bin_acc, width=1 / 15, align="edge", color="#3498DB", alpha=0.7)
    ax.plot([0, 1], [0, 1], "k--")
    ax.set_title(f"{model_name}\nECE={calibration_df.loc[calibration_df['model']==model_name,'ece'].values[0]:.4f}", fontweight="bold")
    ax.set_xlabel("Confidence"); ax.set_ylabel("Accuracy")
    ax.set_xlim(0, 1); ax.set_ylim(0, 1)
plt.tight_layout()
plt.savefig(A01_FIGURE_DIR / "test_reliability_diagram.png", dpi=140, bbox_inches="tight")
plt.show()
print(f"Saved -> {A01_FIGURE_DIR / 'test_reliability_diagram.png'}")

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5.5))
calibration_df.set_index("model")[["nll", "brier", "ece"]].plot(kind="bar", ax=ax)
ax.set_title("NLL / Brier / ECE -- final models on test", fontweight="bold")
plt.xticks(rotation=20, ha="right")
plt.tight_layout()
plt.savefig(A01_FIGURE_DIR / "test_nll_ece_brier_comparison.png", dpi=140, bbox_inches="tight")
plt.show()
print(f"Saved -> {A01_FIGURE_DIR / 'test_nll_ece_brier_comparison.png'}")

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5.5))
x = np.arange(len(FINAL_MODELS))
width = 0.35
ax.bar(x - width / 2, calibration_df["mean_confidence_correct"], width, label="Correct", color="#2ECC71")
ax.bar(x + width / 2, calibration_df["mean_confidence_wrong"], width, label="Wrong", color="#E74C3C")
ax.set_xticks(x); ax.set_xticklabels(FINAL_MODELS, rotation=20, ha="right")
ax.set_ylabel("Mean confidence")
ax.set_title("Mean confidence: correct vs wrong predictions", fontweight="bold")
ax.legend()
plt.tight_layout()
plt.savefig(A01_FIGURE_DIR / "test_confidence_correct_vs_incorrect.png", dpi=140, bbox_inches="tight")
plt.show()
print(f"Saved -> {A01_FIGURE_DIR / 'test_confidence_correct_vs_incorrect.png'}")

## 14. Subgroup analysis

Uses only subgroups genuinely supported by available metadata — here, left vs. right eye
(available in the split CSVs) and class prevalence (already covered per-class in Section 8,
included again here for a single consolidated subgroup table). Other subgroups listed in the
brief (image quality, patient sex/age, preprocessing-failure flags, paired-vs-unpaired eyes)
are **not** computed because this project's manifest does not carry that metadata — adding
them here would mean fabricating a signal, not reporting one.

In [ ]:
test_meta_df = test_df.set_index("image").loc[aligned_test_ids].reset_index()
assert (test_meta_df["image"].to_numpy() == np.array(aligned_test_ids)).all()

subgroup_rows = []
for eye_value in sorted(test_meta_df["eye"].dropna().unique()):
    mask = (test_meta_df["eye"] == eye_value).to_numpy()
    n = int(mask.sum())
    if n == 0:
        continue
    for model_name in FINAL_MODELS:
        preds = model_preds_final[model_name][mask]
        labels_sub = aligned_test_labels[mask]
        qwk_stable = n >= 30  # small-group QWK is unstable; flagged rather than silently reported
        subgroup_rows.append({
            "subgroup": f"eye={eye_value}", "n": n, "model": model_name,
            "qwk": cohen_kappa_score(labels_sub, preds, weights="quadratic") if qwk_stable else np.nan,
            "qwk_stable": qwk_stable,
            "macro_f1": f1_score(labels_sub, preds, average="macro", zero_division=0),
            "accuracy": accuracy_score(labels_sub, preds),
            "mae_grade": float(np.mean(np.abs(preds - labels_sub))),
        })

for class_id in range(NUM_CLASSES):
    mask = (aligned_test_labels == class_id)
    n = int(mask.sum())
    for model_name in FINAL_MODELS:
        preds = model_preds_final[model_name][mask]
        labels_sub = aligned_test_labels[mask]
        qwk_stable = n >= 30
        subgroup_rows.append({
            "subgroup": f"true_class={CLASS_NAMES[class_id]}", "n": n, "model": model_name,
            "qwk": cohen_kappa_score(labels_sub, preds, weights="quadratic") if (qwk_stable and len(set(labels_sub)) > 1) else np.nan,
            "qwk_stable": qwk_stable and len(set(labels_sub)) > 1,
            "macro_f1": f1_score(labels_sub, preds, average="macro", zero_division=0),
            "accuracy": accuracy_score(labels_sub, preds),
            "mae_grade": float(np.mean(np.abs(preds - labels_sub))),
        })

subgroup_performance_df = pd.DataFrame(subgroup_rows)
print(subgroup_performance_df.round(4).to_string(index=False))
subgroup_performance_df.to_csv(A01_LOG_DIR / "subgroup_performance.csv", index=False)
print(f"\nSaved -> {A01_LOG_DIR / 'subgroup_performance.csv'}")
print("\nNote: QWK is left blank (qwk_stable=False) for any subgroup with n<30 or a single "
      "represented class, since QWK is unstable/undefined in those cases - not silently "
      "computed and reported as if reliable.")

## 15. Error analysis table

Clinically-relevant error counts and percentages, per model. "Referable DR" = grades 2-4
(Moderate/Severe/PDR); "No DR" = grade 0.

In [ ]:
clinically_relevant_rows = []
n_test = len(aligned_test_labels)
for model_name in FINAL_MODELS:
    preds = model_preds_final[model_name]
    labels = aligned_test_labels

    is_referable_true = labels >= 2
    is_referable_pred = preds >= 2

    counts = {
        "Mild -> No DR": int(np.sum((labels == 1) & (preds == 0))),
        "Severe -> Moderate": int(np.sum((labels == 3) & (preds == 2))),
        "PDR -> Moderate": int(np.sum((labels == 4) & (preds == 2))),
        "No DR -> referable DR": int(np.sum((labels == 0) & is_referable_pred)),
        "Referable DR -> No DR": int(np.sum(is_referable_true & (preds == 0))),
        "Errors >= 2 grades": int(np.sum(np.abs(preds - labels) >= 2)),
        "High-confidence incorrect (>=0.80)": int(np.sum((final_test_probs[model_name].max(axis=1) >= 0.80) & (preds != labels))),
    }
    for error_type, count in counts.items():
        clinically_relevant_rows.append({
            "model": model_name, "error_type": error_type, "count": count, "percent_of_test_set": 100 * count / n_test,
        })

clinically_relevant_error_summary_df = pd.DataFrame(clinically_relevant_rows)
print(clinically_relevant_error_summary_df.round(2).to_string(index=False))
clinically_relevant_error_summary_df.to_csv(A01_LOG_DIR / "clinically_relevant_error_summary.csv", index=False)
print(f"\nSaved -> {A01_LOG_DIR / 'clinically_relevant_error_summary.csv'}")

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5.5))
overview_metrics = ["qwk", "macro_f1", "balanced_accuracy", "accuracy"]
final_test_metrics_df[overview_metrics].plot(kind="bar", ax=ax)
ax.set_title("Overall test metric comparison -- final models", fontweight="bold")
plt.xticks(rotation=20, ha="right")
plt.tight_layout()
plt.savefig(A01_FIGURE_DIR / "overall_test_metric_comparison.png", dpi=140, bbox_inches="tight")
plt.show()
print(f"Saved -> {A01_FIGURE_DIR / 'overall_test_metric_comparison.png'}")

In [ ]:
per_class_f1_pivot = final_test_per_class_metrics_df.pivot(index="class", columns="model", values="f1").reindex(CLASS_NAMES)

fig, ax = plt.subplots(figsize=(10, 5.5))
per_class_f1_pivot.plot(kind="bar", ax=ax)
ax.set_ylabel("F1 score")
ax.set_title("Per-class F1 -- final models (test set)", fontweight="bold")
plt.xticks(rotation=20, ha="right")
plt.tight_layout()
plt.savefig(A01_FIGURE_DIR / "per_class_f1_test_comparison.png", dpi=140, bbox_inches="tight")
plt.show()
print(f"Saved -> {A01_FIGURE_DIR / 'per_class_f1_test_comparison.png'}")

## 16. Final comparison rule

Hierarchy: test QWK -> test macro-F1 -> balanced accuracy -> calibration -> ordinal error
severity -> simplicity/computational cost. **This ranking is computed directly from the test
results above, not assumed in advance** — M13 is not automatically declared best because it
was selected on validation; if the test data show M12 performing equally well, or DeiT/
EfficientNet performing better, or improvements within the overlapping confidence intervals
found in Section 11, those are all scientifically acceptable outcomes and are what gets
reported here.

In [ ]:
ranking_df = final_test_metrics_df.sort_values(
    ["qwk", "macro_f1", "balanced_accuracy", "ece"], ascending=[False, False, False, True]
)
print("Final ranking (QWK -> macro-F1 -> balanced accuracy -> calibration):")
print(ranking_df[["qwk", "macro_f1", "balanced_accuracy", "ece"]].round(4).to_string())

best_model_by_rule = ranking_df.index[0]
print(f"\nTop-ranked model by the stated hierarchy: {best_model_by_rule}")

m13_vs_m12_qwk_row = paired_bootstrap_model_differences_df[
    (paired_bootstrap_model_differences_df["model_a"] == "M13-D")
    & (paired_bootstrap_model_differences_df["model_b"] == "M12")
    & (paired_bootstrap_model_differences_df["metric"] == "qwk")
]
if len(m13_vs_m12_qwk_row) > 0:
    row = m13_vs_m12_qwk_row.iloc[0]
    if row["ci_excludes_zero"]:
        direction = "M13-D outperforms M12" if row["difference"] > 0 else "M12 outperforms M13-D"
        print(f"M13-D vs M12 QWK difference is supported by the confidence interval: {direction} "
              f"(diff={row['difference']:.4f}, CI=[{row['ci_lower']:.4f}, {row['ci_upper']:.4f}]).")
    else:
        print(f"M13-D vs M12 QWK difference is NOT supported by the confidence interval "
              f"(diff={row['difference']:.4f}, CI=[{row['ci_lower']:.4f}, {row['ci_upper']:.4f}], includes zero) "
              "- the simpler M12 may be preferable given comparable performance.")

## 17. Computational comparison

For M13, note explicitly: inference requires running **every constituent model**, so any
performance improvement over a single model comes with proportionally greater computational
cost — a genuine trade-off, not a free win.

In [ ]:
def count_model_parameters(model_name: str):
    """Best-effort parameter count from the model's own saved config.json, where available."""
    log_dir = FROZEN_MODELS[model_name]["log_dir"]
    config_path = log_dir / "config.json"
    if config_path.exists():
        with open(config_path) as f:
            cfg = json.load(f)
        return cfg.get("total_parameters")
    return None


complexity_rows = []
for model_name in ["EfficientNet-B4", "DeiT-III", "M12"]:
    checkpoint_dir = FROZEN_MODELS[model_name]["log_dir"].parent.parent / "checkpoints" if False else None
    complexity_rows.append({
        "model": model_name,
        "parameter_count": count_model_parameters(model_name),
        "input_resolution": FROZEN_MODELS[model_name]["input_size"],
        "inference_passes_per_image": 1 if model_name != "M12" else 3,
        "ensemble_member_count": 1,
        "relative_deployment_complexity": "single model",
    })

m13_total_passes = sum(
    (1 if name != "M12" else 3) for name in m13_constituent_models
)
complexity_rows.append({
    "model": "M13-D",
    "parameter_count": sum(
        (count_model_parameters(name) or 0) for name in m13_constituent_models
    ) or None,
    "input_resolution": "n/a (multiple constituent resolutions)",
    "inference_passes_per_image": m13_total_passes,
    "ensemble_member_count": len(m13_constituent_models),
    "relative_deployment_complexity": f"ensemble of {len(m13_constituent_models)} models ({', '.join(m13_constituent_models)})",
})

model_complexity_comparison_df = pd.DataFrame(complexity_rows)
print(model_complexity_comparison_df.to_string(index=False))
model_complexity_comparison_df.to_csv(A01_LOG_DIR / "model_complexity_comparison.csv", index=False)
print(f"\nSaved -> {A01_LOG_DIR / 'model_complexity_comparison.csv'}")
print(
    f"\nM13-D requires {m13_total_passes} forward passes per test image across its "
    f"{len(m13_constituent_models)} constituent models, vs. 1 pass for a single standalone model "
    "- any QWK/macro-F1 gain from Section 16 should be weighed against this multiplier."
)

## 18. Final notebook conclusion

*To be completed after this notebook has actually been run — do not write the conclusion
before the results above are generated.*

To be answered from the actual results:

- Which model achieved the best test QWK?
- Which achieved the best macro-F1?
- Did M13 improve over M12 (Section 11's paired comparison, not just the point estimate)?
- Were improvements supported by paired confidence intervals excluding zero?
- Which classes improved (Section 8's per-class table, particularly Mild/Severe/PDR)?
- Which errors remained persistent (Section 15's clinically-relevant error table)?
- Was calibration improved on test, not just on validation (Section 13)?
- Is the ensemble improvement (if any) worth its computational complexity (Section 17)?
- What limitations affect interpretation — patient-level bootstrap assumptions, the
  non-pristine test-set caveat from Section 1, and subgroups not computed due to missing
  metadata (Section 14)?